## CSCI-115  Finance Data ingestion for Stock Screener ##


This notebook is mainly for gathering Finance data from Yahoo finance and Finance data from https://site.financialmodelingprep.com/developer

In [1]:
#Load Library
import ta
from ta import add_all_ta_features
import pandas as pd
import numpy as np
import yfinance as yf
import requests

In [ ]:
#Check if TA library install correctly
#import sys
#print(sys.executable)

#pip install yfinance
#!"C:/Users/siris/micromamba/micromamba/envs/cs109b/python.exe" -m pip install ta
#!pip install ta

#
#print(" TA package is working!")

**Load yahoo finance data**

In [3]:
import pandas as pd
import yfinance as yf

# === 1. Load Tickers from CSV ===
# Make sure your CSV file has a column named 'Ticker'
sp500_df = pd.read_csv("S&P500_list.csv")
tickers = sp500_df['Symbol'].dropna().astype(str).unique().tolist()
tickers = [t.replace('.', '-') for t in tickers]  

print(f" Loaded {len(tickers)} S&P 500 tickers")

# === 2. Set Date Range ===
start_date = "2019-01-01"
end_date = "2025-09-30"

# === 3. Download Data ===
print(f"  Downloading OHLCV data from {start_date} to {end_date} for {len(tickers)} tickers...")
raw_data = yf.download(
    tickers,
    start=start_date,
    end=end_date,
    interval="1d",
    group_by='ticker',
    threads=True
)

# === 4. Save Raw Wide Format ===
raw_data.to_csv("sp500_raw_data.csv")
print(" Saved: sp500_raw_data.csv")

# === 5. Convert to Long Format ===
if isinstance(raw_data.columns, pd.MultiIndex):
    long_data = []

    for ticker in tickers:
        try:
            df = raw_data[ticker].copy()
            df['Ticker'] = ticker
            df['Date'] = df.index
            long_data.append(df)
        except KeyError:
            print(f" Ticker {ticker} missing or malformed.")
            continue

    long_df = pd.concat(long_data, axis=0).reset_index(drop=True)

    # Drop missing Close rows
    long_df = long_df.dropna(subset=['Close'])

    # Reorder columns
    ordered_cols = ['Date', 'Ticker', 'Open', 'High', 'Low', 'Close', 'Volume']
    long_df = long_df[[col for col in ordered_cols if col in long_df.columns]]

    # Save long format
    long_df.to_csv("sp500_long_data.csv", index=False)
    print(" Saved: sp500_long_data.csv")

    # Summary
    print(f" Successfully downloaded data for {long_df['Ticker'].nunique()} tickers")
else:
    print("❗ Unexpected data format — raw_data does not use MultiIndex. Check the ticker list or try smaller batch.")


 Loaded 504 S&P 500 tickers


C:\Users\siris\AppData\Local\Temp\ipykernel_28736\4038389131.py:18: FutureWarning: YF.download() has changed argument auto_adjust default to True
  raw_data = yf.download(
[**********************70%*********              ]  351 of 504 completedHTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: HES"}}}
[*********************100%***********************]  504 of 504 completed

4 Failed downloads:
['JNPR', 'ANSS', 'HES']: YFTzMissingError('possibly delisted; no timezone found')
['PARA']: YFPricesMissingError('possibly delisted; no price data found  (1d 2019-01-01 -> 2025-09-30) (Yahoo error = "No data found, symbol may be delisted")')


 Saved: sp500_raw_data.csv
 Saved: sp500_long_data.csv
 Successfully downloaded data for 500 tickers


In [4]:
long_df.head()

Price,Date,Ticker,Open,High,Low,Close,Volume
0,2019-01-02,MSFT,93.317966,95.380239,92.746153,94.789680,35329300.0
1,2019-01-03,MSFT,93.833554,93.917923,91.115098,91.302582,42579100.0
2,2019-01-04,MSFT,93.477317,96.092658,92.736772,95.548965,44060600.0
3,2019-01-07,MSFT,95.277162,96.805119,94.658483,95.670868,35656100.0
4,2019-01-08,MSFT,96.589491,97.461271,95.342750,96.364517,31514400.0


## Add technical factor

In [8]:
#This section of code is to calculate technical data using TA library

# 1. Load S&P500 Long datayour data
df = pd.read_csv("sp500_long_data.csv", parse_dates=["Date"])

# 2. Ensure data is sorted by Ticker and Date
df = df.sort_values(by=["Ticker", "Date"])

# 3. Create an empty list to hold processed frames
enhanced_frames = []

# 4. Apply technical indicators per ticker
for ticker, group in df.groupby("Ticker"):
    group = group.copy()

    # Add technical indicators using `ta` library
    group = add_all_ta_features(
        df=group,
        open="Open",
        high="High",
        low="Low",
        close="Close",
        volume="Volume",
        fillna=True  # fill missing values where needed
    )

    enhanced_frames.append(group)

# 5. Combine all tickers back into one DataFrame
df_ta = pd.concat(enhanced_frames, axis=0).reset_index(drop=True)

# 6. Save result to new CSV
df_ta.to_csv("sp500_long_data_with_complete_ta.csv", index=False)
print(" TA-enhanced data saved to: sp500_long_data_with_ta.csv")


c:\Users\siris\micromamba\micromamba\envs\cs109b\Lib\site-packages\numpy\core\fromnumeric.py:59: RuntimeWarning: invalid value encountered in accumulate
  return bound(*args, **kwds)
c:\Users\siris\micromamba\micromamba\envs\cs109b\Lib\site-packages\numpy\core\fromnumeric.py:59: RuntimeWarning: invalid value encountered in accumulate
  return bound(*args, **kwds)


 TA-enhanced data saved to: sp500_long_data_with_ta.csv


In [13]:
pd.set_option('display.max_columns', None)

df_ta.head(10)


,Date,Ticker,Open,High,Low,Close,Volume,volume_adi,volume_obv,volume_cmf,volume_fi,volume_em,volume_sma_em,volume_vpt,volume_vwap,volume_mfi,volume_nvi,volatility_bbm,volatility_bbh,volatility_bbl,volatility_bbw,volatility_bbp,volatility_bbhi,volatility_bbli,volatility_kcc,volatility_kch,volatility_kcl,volatility_kcw,volatility_kcp,volatility_kchi,volatility_kcli,volatility_dcl,volatility_dch,volatility_dcm,volatility_dcw,volatility_dcp,volatility_atr,volatility_ui,trend_macd,trend_macd_signal,trend_macd_diff,trend_sma_fast,trend_sma_slow,trend_ema_fast,trend_ema_slow,trend_vortex_ind_pos,trend_vortex_ind_neg,trend_vortex_ind_diff,trend_trix,trend_mass_index,trend_dpo,trend_kst,trend_kst_sig,trend_kst_diff,trend_ichimoku_conv,trend_ichimoku_base,trend_ichimoku_a,trend_ichimoku_b,trend_stc,trend_adx,trend_adx_pos,trend_adx_neg,trend_cci,trend_visual_ichimoku_a,trend_visual_ichimoku_b,trend_aroon_up,trend_aroon_down,trend_aroon_ind,trend_psar_up,trend_psar_down,trend_psar_up_indicator,trend_psar_down_indicator,momentum_rsi,momentum_stoch_rsi,momentum_stoch_rsi_k,momentum_stoch_rsi_d,momentum_tsi,momentum_uo,momentum_stoch,momentum_stoch_signal,momentum_wr,momentum_ao,momentum_roc,momentum_ppo,momentum_ppo_signal,momentum_ppo_hist,momentum_pvo,momentum_pvo_signal,momentum_pvo_hist,momentum_kama,others_dr,others_dlr,others_cr
0,2019-01-02,A,63.325221,63.391879,62.182514,62.553894,2113300,-8.153663e+05,2113300,-0.385826,0.000000e+00,0.000000,0.000000,0.000000,62.709429,50.000000,1000.000000,62.553894,62.553894,62.553894,0.000000,0.000000,0.0,0.0,62.709429,63.918795,61.500063,3.857046,0.435696,0.0,0.0,62.182514,63.391879,62.787196,1.933318,0.307087,0.000000,0.0,0.000000,0.000000,0.000000,62.553894,62.553894,62.553894,62.553894,0.000000,0.000000,0.000000,-45.783121,1.000000,53.580347,-461.365630,-461.365630,0.000000,62.787196,62.787196,62.787196,62.787196,0.0,0.0,0.0,0.0,0.000000,115.711653,115.12776,0.0,0.0,0.0,59.040058,73.028729,0.0,0.0,100.000000,0.0,0.0,0.0,0.000000,0.000000,30.708695,30.708695,-69.291305,0.000000,0.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,62.553894,0.000000,0.000000,0.000000
1,2019-01-03,A,62.401532,62.639596,59.040058,60.249428,5383900,-2.581511e+06,-3270600,-0.344330,-1.240702e+07,-130.196120,-130.196120,-198341.222718,61.225502,0.000000,1000.000000,61.401661,63.706127,59.097195,7.506202,0.250000,0.0,0.0,61.676228,64.080680,59.271776,7.797013,0.203300,0.0,0.0,59.040058,63.391879,61.215969,7.087464,0.277900,0.000000,0.0,-0.183832,-0.036766,-0.147066,61.401661,61.401661,62.199361,62.383193,0.007942,0.075616,-0.067674,-0.007195,2.293054,54.732581,-471.287192,-466.326411,-4.960781,61.215969,61.215969,61.215969,61.215969,0.0,0.0,0.0,0.0,-66.666667,115.711653,115.12776,0.0,4.0,-4.0,59.040058,73.028729,0.0,0.0,0.000000,0.0,0.0,0.0,-100.000000,25.148547,27.789962,29.249328,-72.210038,0.000000,0.0,-0.294682,-0.058936,-0.235746,11.076000,2.215200,8.860800,61.441152,-3.683969,-3.753541,-3.683969
2,2019-01-04,A,61.030268,62.801470,61.030268,62.334866,3123700,-1.103624e+06,-146900,-0.103911,-9.703973e+06,61.013792,-34.591164,-90219.333182,61.469622,37.253199,1034.613404,61.712729,63.789863,59.635596,6.731621,0.649758,0.0,0.0,61.802664,63.996032,59.609295,7.097974,0.621321,0.0,0.0,59.040058,63.391879,61.215969,7.051740,0.757110,0.000000,0.0,-0.159405,-0.061294,-0.098111,61.712729,61.712729,62.220208,62.379613,0.070187,0.099182,-0.028994,-0.012377,3.523664,54.421512,-468.608669,-467.087164,-1.521506,61.215969,61.215969,61.215969,61.215969,0.0,0.0,0.0,0.0,21.806062,115.711653,115.12776,0.0,4.0,-4.0,59.040058,73.028729,1.0,0.0,49.355978,0.0,0.0,0.0,-98.009009,44.760647,75.711001,44.736552,-24.288999,0.000000,0.0,-0.255541,-0.098257,-0.157284,11.690916,4.110343,7.580573,61.834976,3.461340,3.402783,-0.350144
3,2019-01-07,A,62.506276,64.210820,62.477709,63.658508,3235100,6.953763e+04,3088200,0.005019,-7.705960e+06,76.522140,2.446604,-21523.964082,61.931770,55.003480,1034.613404,62.199174,64.664008,59.734340,7.9256

In [ ]:
# Define columns to keep
columns_to_keep = [
    'Date', 'Ticker', 'Open', 'High', 'Low', 'Close', 'Volume',
    # Trend
    'trend_sma_fast', 'trend_sma_slow', 'trend_macd', 'trend_macd_signal', 'trend_adx',
    # Momentum
    'momentum_rsi', 'momentum_stoch_k', 'momentum_wr',
    # Volatility
    'volatility_bbm', 'volatility_bbh', 'volatility_bbl', 'volatility_atr',
    # Volume
    'volume_obv', 'volume_mfi',
    # Others
    'others_dlr'
]

# Keep only those columns above
df_ta_core = df_ta.drop(columns=[col for col in df_ta.columns if col not in columns_to_keep])


# Check the result
print(" Shape after filtering:", df_ta_core.shape)
print(df_ta_core.head())

# Save result to new CSV
df_ta_core.to_csv("sp500_long_data_with_ta_core.csv", index=False)
df_ta_core.to_parquet("sp500_long_data_with_ta_core.parquet", index=False)
print(" TA-core data saved to: sp500_long_data_with_ta_core.csv and clean_stock_ta_features.parquet")

 Shape after filtering: (836242, 21)
        Date Ticker       Open       High        Low      Close   Volume  \
0 2019-01-02      A  63.325221  63.391879  62.182514  62.553894  2113300   
1 2019-01-03      A  62.401532  62.639596  59.040058  60.249428  5383900   
2 2019-01-04      A  61.030268  62.801470  61.030268  62.334866  3123700   
3 2019-01-07      A  62.506276  64.210820  62.477709  63.658508  3235100   
4 2019-01-08      A  64.363188  64.953592  63.515678  64.591736  1578100   

   volume_obv  volume_mfi  volatility_bbm  volatility_bbh  volatility_bbl  \
0     2113300   50.000000       62.553894       62.553894       62.553894   
1    -3270600    0.000000       61.401661       63.706127       59.097195   
2     -146900   37.253199       61.712729       63.789863       59.635596   
3     3088200   55.003480       62.199174       64.664008       59.734340   
4     4666300   60.528039       62.677686       65.597260       59.758113   

   volatility_atr  trend_macd  trend_macd_s

There are total 93 technical indicator. We narrow down to the indicators which are necessary as following

| **Type**       | **Indicator**                                         | **Column Pattern (from `ta`)**                             | **Meaning / Interpretation**                                                                                                                               |
| -------------- | ----------------------------------------------------- | ---------------------------------------------------------- | ---------------------------------------------------------------------------------------------------------------------------------------------------------- |
| **Trend**      | **20-day SMA (Simple Moving Average)**                | `'trend_sma_fast'`                                         | Average closing price over the past 20 days. Tracks short-term trend direction. Price above 20SMA → short-term uptrend.                                    |
|                | **50-day SMA (Simple Moving Average)**                | `'trend_sma_slow'`                                         | Average closing price over the past 50 days. Represents medium-term trend. Used with 20SMA to detect crossovers (e.g., “Golden Cross”).                    |
|                | **MACD line (Moving Average Convergence Divergence)** | `'trend_macd'`                                             | Difference between 12-day EMA and 26-day EMA. Positive = bullish momentum, negative = bearish.                                                             |
|                | **MACD signal**                                       | `'trend_macd_signal'`                                      | 9-day EMA of the MACD line. When MACD crosses above the signal → bullish; below → bearish.                                                                 |
|                | **ADX (Average Directional Index)**                   | `'trend_adx'`                                              | Measures *trend strength* (0–100). ADX > 25 = strong trend; ADX < 20 = weak/sideways market.                                                               |
| **Momentum**   | **RSI (Relative Strength Index, 14)**                 | `'momentum_rsi'`                                           | Oscillator (0–100) that measures price momentum. RSI > 70 = overbought; RSI < 30 = oversold.                                                               |
|                | **Stochastic %K**                                     | `'momentum_stoch_k'`                                       | Compares current price to recent highs/lows (0–100). Values above 80 = overbought; below 20 = oversold.                                                    |
|                | **Williams %R**                                       | `'momentum_wr'`                                            | Shows how close the current price is to the recent high/low range (-100 to 0). Below -80 = oversold; above -20 = overbought.                               |
| **Volatility** | **Bollinger Bands (BBM, BBH, BBL)**                   | `'volatility_bbm'`, `'volatility_bbh'`, `'volatility_bbl'` | Measures volatility using SMA ± 2 standard deviations. When price touches upper band → overbought; lower band → oversold. Band width indicates volatility. |
|                | **ATR (Average True Range)**                          | `'volatility_atr'`                                         | Measures average daily price range over a period. Higher ATR = more volatility (larger moves). Useful for stop-loss sizing.                                |
| **Volume**     | **OBV (On-Balance Volume)**                           | `'volume_obv'`                                             | Cumulative total of volume added/subtracted based on price movement. Rising OBV = buying pressure; falling OBV = selling pressure.                         |
|                | **MFI (Money Flow Index)**                            | `'volume_mfi'`                                             | Combines price and volume into an oscillator (0–100). MFI > 80 = overbought; MFI < 20 = oversold. Confirms strength of moves.                              |
| **Others**     | **Daily Log Return**                                  | `'others_dlr'`                                             | Natural log of price ratio (`ln(close/previous_close)`). Measures daily percentage change for modeling returns and volatility.                             |


| **Category**   | **Purpose**                                                 | **Example Use in Screener**                                      |
| -------------- | ----------------------------------------------------------- | ---------------------------------------------------------------- |
| **Trend**      | Detect overall direction (uptrend/downtrend).               | Select stocks where price > 20SMA and MACD > 0 (bullish).        |
| **Momentum**   | Detect buying/selling strength and reversal zones.          | Find stocks with RSI < 30 (oversold, potential rebound).         |
| **Volatility** | Gauge market risk and price fluctuation.                    | Filter out extremely volatile stocks using low ATR.              |
| **Volume**     | Confirm whether trends are backed by strong trading volume. | Only consider buy signals when OBV and MFI are rising.           |
| **Others**     | Track price returns for performance or normalization.       | Calculate Sharpe ratio or model price change as target variable. |


Extract data from financial 

https://site.financialmodelingprep.com/developer



## Income Statement ##


In [10]:
#Income statement
#  Your API Key
API_KEY = "VXTY4Jz8jYbQ9R4PjjkmqJ8VLROXlC3Z"

#  Correct endpoint (stable)
#symbol = "AAPL"
#url = f"https://financialmodelingprep.com/stable/income-statement?symbol={symbol}&apikey={API_KEY}"

symbols = ["AAPL", "MSFT","NVDA"]
all_data = []

for symbol in symbols:
    url = f"https://financialmodelingprep.com/stable/income-statement?symbol={symbol}&apikey={API_KEY}"
    response = requests.get(url)

    if response.status_code == 200:
        df = pd.DataFrame(response.json())
        df["Ticker"] = symbol
        all_data.append(df)
    else:
        print(f"❌ Failed for {symbol}: {response.status_code}")

final_df = pd.concat(all_data)
final_df.to_csv("SP500_income_statements.csv", index=False)

In [14]:
#Financial ratio


# Correct endpoint (stable)
#symbol = "AAPL"
#url = f"https://financialmodelingprep.com/stable/income-statement?symbol={symbol}&apikey={API_KEY}"

symbols = ["AAPL", "MSFT","NVDA"]
all_data = []

for symbol in symbols:
    #url = f"https://financialmodelingprep.com/stable/income-statement?symbol={symbol}&apikey={API_KEY}"
    url = f"https://financialmodelingprep.com/stable/ratios?symbol={symbol}&apikey={API_KEY}"
    response = requests.get(url)

    if response.status_code == 200:
        df = pd.DataFrame(response.json())
        df["Ticker"] = symbol
        all_data.append(df)
    else:
        print(f"❌ Failed for {symbol}: {response.status_code}")

final_df = pd.concat(all_data)
final_df.to_csv("SP500_finance_ratio.csv", index=False)

In [15]:
#Financial ratio
#  API Key
API_KEY = "VXTY4Jz8jYbQ9R4PjjkmqJ8VLROXlC3Z"

# Correct endpoint (stable)
#symbol = "AAPL"
#url = f"https://financialmodelingprep.com/stable/income-statement?symbol={symbol}&apikey={API_KEY}"

symbols = ["AAPL", "MSFT","NVDA"]
all_data = []

for symbol in symbols:
    #url = f"https://financialmodelingprep.com/stable/income-statement?symbol={symbol}&apikey={API_KEY}"
    url = f"https://financialmodelingprep.com/stable/ratios?symbol={symbol}&apikey={API_KEY}"
    response = requests.get(url)

    if response.status_code == 200:
        df = pd.DataFrame(response.json())
        df["Ticker"] = symbol
        all_data.append(df)
    else:
        print(f"❌ Failed for {symbol}: {response.status_code}")

final_df = pd.concat(all_data)
final_df.to_csv("SP500_finance_ratio.csv", index=False)

In [ ]:
#This one is not working yet. Need to pay for data subscription. This is for code testing only

# =======================
# 🔧 Configuration
# =======================
API_KEY = "VXTY4Jz8jYbQ9R4PjjkmqJ8VLROXlC3Z"
BASE_URL = "https://financialmodelingprep.com/stable"
OUTPUT_DIR = "fmp_output"
symbols = ["AAPL", "MSFT", "NVDA"]

# List of API endpoints to fetch
ENDPOINTS = {
    "key-metrics": "/key-metrics/{symbol}",
    "income-statement": "/income-statement/{symbol}",
    "balance-sheet": "/balance-sheet-statement/{symbol}",
    "cash-flow": "/cash-flow-statement/{symbol}",
    "profile": "/profile/{symbol}",
    "ratios": "/ratios/{symbol}",
    "enterprise-values": "/enterprise-values/{symbol}"
}

# =======================
#  Download Function
# =======================
def fetch_and_save(symbol, endpoint_name, endpoint_path):
    url = f"{BASE_URL}{endpoint_path.format(symbol=symbol)}?apikey={API_KEY}"
    try:
        response = requests.get(url)
        if response.status_code == 200:
            data = response.json()
            if isinstance(data, dict):  # Handle 'profile' which returns a single dict
                data = [data]
            df = pd.DataFrame(data)
            df["Ticker"] = symbol

            # Ensure the folder exists
            endpoint_folder = os.path.join(OUTPUT_DIR, endpoint_name)
            os.makedirs(endpoint_folder, exist_ok=True)

            output_path = os.path.join(endpoint_folder, f"{symbol}.csv")
            df.to_csv(output_path, index=False)
            print(f"✅ Saved: {output_path}")
        else:
            print(f"❌ Failed for {symbol} at {endpoint_name}: {response.status_code}")
    except Exception as e:
        print(f"❌ Error fetching {symbol} from {endpoint_name}: {e}")

# =======================
#  Run the downloads
# =======================
for symbol in symbols:
    for endpoint_name, endpoint_path in ENDPOINTS.items():
        fetch_and_save(symbol, endpoint_name, endpoint_path)


❌ Failed for AAPL at key-metrics: 404
❌ Failed for AAPL at income-statement: 404
❌ Failed for AAPL at balance-sheet: 404
❌ Failed for AAPL at cash-flow: 404
❌ Failed for AAPL at profile: 404
❌ Failed for AAPL at ratios: 404
❌ Failed for AAPL at enterprise-values: 404
❌ Failed for MSFT at key-metrics: 404
❌ Failed for MSFT at income-statement: 404
❌ Failed for MSFT at balance-sheet: 404
❌ Failed for MSFT at cash-flow: 404
❌ Failed for MSFT at profile: 404
❌ Failed for MSFT at ratios: 404
❌ Failed for MSFT at enterprise-values: 404
❌ Failed for NVDA at key-metrics: 404
❌ Failed for NVDA at income-statement: 404
❌ Failed for NVDA at balance-sheet: 404
❌ Failed for NVDA at cash-flow: 404
❌ Failed for NVDA at profile: 404
❌ Failed for NVDA at ratios: 404
❌ Failed for NVDA at enterprise-values: 404


In [21]:
import os
print(os.getcwd())

c:\Users\siris\OneDrive\CSCI-E 115_od\Project\data
